# Autoencoders Accuracy

In [ ]:
import sys
with open("./../../../PATHS.txt") as file:
  paths = file.read().splitlines()
sys.path.extend(paths)

In [ ]:
import os
import json
import numpy as np

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D

colors = matplotlib.rcParams["axes.prop_cycle"].by_key()["color"]
markers = Line2D.filled_markers[1:]

In [ ]:
from dd_nm_rom import ops

In [ ]:
dim_ranges = {
  "interior": {
    "latent_dim": {
      "start": 8,
      "stop": 21,
      "step": 4
    },
    "row_nonzero": 12
  },
  "port": {
    "latent_dim": {
      "start": 6,
      "stop": 11,
      "step": 2
    },
    "row_nonzero": 8
  }
}
elements = ["interior", "port"]
prefix = "/g/g92/zanardi1/Workspace/Codes/DD-NM-ROM/run/unsteady/dset.2by2/figures/"

In [ ]:
def_styles = {
  "interior": lambda index: dict(
    color=colors[index],
    marker=markers[index],
    markersize=7,
    markerfacecolor=colors[index],
    markeredgecolor=colors[index],
    linestyle=""
  ),
  "port": lambda index: dict(
    color=colors[index],
    linestyle="--"
  )
}

Generate all configurations

In [ ]:
dims, cfgs = {}, []
for element in elements:
  dims[element] = ops.generate_combs([
    np.arange(**dim_ranges[element]["latent_dim"]),
    np.array([dim_ranges[element]["row_nonzero"]])
  ])
  cfgs.append(np.arange(len(dims[element])))
cfgs = ops.generate_combs(cfgs)

Loop over configurations

In [ ]:
def read_stats(tag):
  filename = prefix + f"/{tag}/stats_mean.json"
  with open(filename) as file:
    stats = json.load(file)
  error = stats["error"]["srpc"]["mean"]
  speedup = stats["speedup"]["srpc"]["total"]["mean"]
  converged = stats["converged"]["srpc"]
  return error, speedup, converged

In [ ]:
stats = {}
for cfg in cfgs:
  # Set tag
  key, tag = [], []
  for (e, element) in enumerate(elements):
    ld, rnz = dims[element][cfg[e]]
    key.append(ld)
    tag.append(f"{element}_ld_{ld}_rnz_{rnz}")
  tag = "_".join(tag)
  # Read stats
  error, speedup, converged = read_stats(tag)
  key.append(converged)
  stats[tuple(key)] = (speedup, error)

Plot Pareto front

In [ ]:
def plot_pareto_front(styles, filename):
  fig, ax = plt.subplots()
  for element in ("port", "interior"):
    for ld, style in styles[element].items():
      x = style.pop("x")
      y = style.pop("y")
      ax.plot(x, y, **style, label=f"{element}-{ld}")
  ax.set_xlabel("Speedup")
  ax.set_ylabel("Error")
  ax.invert_xaxis()
  ax.grid()
  ax.legend()
  plt.savefig(filename, bbox_inches="tight", pad_inches=0.1)
  plt.show()

In [ ]:
pstyles = {}
for (e, element) in enumerate(elements):
  pstyles[element] = {}
  i, x, y = 0, [], []
  for istats in stats.keys():
    ld = istats[e]
    x.append(stats[istats][0])
    y.append(stats[istats][1])
    if (ld not in pstyles[element]):
      pstyles[element][ld] = def_styles[element](i)
      pstyles[element][ld]["x"] = []
      pstyles[element][ld]["y"] = []
      i += 1
    pstyles[element][ld]["x"].append(stats[istats][0])
    pstyles[element][ld]["y"].append(stats[istats][1])

In [ ]:
path = prefix + "/global/"
os.makedirs(path)
plot_pareto_front(pstyles, path+"/pareto.png")